<a href="https://colab.research.google.com/github/Milu2662/BI-PYMES-ML/blob/main/notebook/02_data_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BI-PYMES-ML — DATA QUALITY

In [2]:
!git clone https://github.com/Milu2662/BI-PYMES-ML.git /content/BI-PYMES-ML

Cloning into '/content/BI-PYMES-ML'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 59 (delta 23), reused 29 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 131.72 KiB | 6.27 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [3]:
from pathlib import Path
import sys
candidates=[Path("/content/BI-PYMES-ML"),Path("/content/SME-DataTrust-BI"),Path.cwd()]
PROJECT_DIR=next((p for p in candidates if (p/"src").exists()),Path.cwd())
if not (PROJECT_DIR/"src").exists():
    raise FileNotFoundError("No se encontró src/. Ajusta PROJECT_DIR a la raíz del repositorio.")
sys.path.insert(0,str(PROJECT_DIR))
for d in ["data/raw","data/processed","data/experimental","results/figures","results/metrics"]:
    (PROJECT_DIR/d).mkdir(parents=True,exist_ok=True)
print("Proyecto:",PROJECT_DIR)

Proyecto: /content/BI-PYMES-ML


In [5]:
from google.colab import drive
from pathlib import Path
import shutil

# Conectar Google Drive
drive.mount('/content/drive')

# Archivo almacenado en Drive
source = Path(
    "/content/drive/MyDrive/BI-PYMES-ML/retail_lines.pkl"
)

# Destino dentro del proyecto
destination = PROJECT_DIR / "data/processed/retail_lines.pkl"

destination.parent.mkdir(parents=True, exist_ok=True)

if not source.exists():
    raise FileNotFoundError("No se encontró el archivo en Google Drive.")

shutil.copy2(source, destination)

print("Datos recuperados correctamente.")
print("Archivo:", destination)
print("Proyecto:",PROJECT_DIR)

Mounted at /content/drive
Datos recuperados correctamente.
Archivo: /content/BI-PYMES-ML/data/processed/retail_lines.pkl
Proyecto: /content/BI-PYMES-ML


## 2. Perfilamiento de calidad
Los valores negativos, cancelaciones y registros poco frecuentes no se eliminan automáticamente: pueden tener significado comercial.

In [6]:
import pandas as pd, json
from src.preprocessing import quality_profile
from src.reliability import quality_dimensions
p=PROJECT_DIR/"data/processed/retail_lines.pkl"
if not p.exists(): raise FileNotFoundError("Ejecuta primero Notebook 01.")
df=pd.read_pickle(p)
profile,checks=quality_profile(df); dimensions=quality_dimensions(df)
display(profile); print(checks); print(dimensions)
profile.to_csv(PROJECT_DIR/"results/metrics/data_quality_profile.csv")
with open(PROJECT_DIR/"results/metrics/quality_dimensions.json","w") as f: json.dump(dimensions,f,indent=2)

,dtype,missing_count,missing_pct,unique_count
InvoiceNo,string,0,0.000,53628
StockCode,object,0,0.000,5305
Description,object,4382,0.411,5698
Quantity,int64,0,0.000,1057
InvoiceDate,datetime64[ns],0,0.000,47635
UnitPrice,float64,0,0.000,2807
CustomerID,float64,243007,22.767,5942
Country,object,0,0.000,43
SourceSheet,object,0,0.000,2
Revenue,float64,0,0.000,9113


{'rows': 1067371, 'columns': 10, 'exact_duplicates': 12133, 'quantity_negative': 22950, 'quantity_zero': 0, 'price_negative': 5, 'invalid_dates': 0, 'cancellation_invoices': 19494}
{'completeness': 0.976822585586455, 'uniqueness': 0.9886328183921055, 'rule_based_validity': 0.9999984385310574, 'observed_quality_index': 98.73184113115308}
